In [3]:
# Cell 1 — Import
import sys
import json
import os
sys.path.append('../')
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv('../.env')

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)
print("Kết nối Neo4j thành công")

Kết nối Neo4j thành công


In [4]:
# Cell 2 — Đọc regulation JSON
with open('../data/raw/sop/regulation_graph.json', 'r') as f:
    reg = json.load(f)

print(f"Document: {reg['document']['title']}")
print(f"Activities: {len(reg['activities'])}")
print(f"Sequences:  {len(reg['sequences'])}")
print(f"Conditions: {len(reg['conditions'])}")
print(f"Roles:      {len(reg['roles'])}")

Document: Directive 2008/48/EC on Credit Agreements for Consumers
Activities: 13
Sequences:  7
Conditions: 4
Roles:      3


In [6]:
# Cell 3 — Tạo Document node
with driver.session() as session:
    d = reg['document']
    session.run("""
        MERGE (doc:Document {id: $id})
        SET doc.title          = $title,
            doc.issuer         = $issuer,
            doc.effective_date = $effective_date,
            doc.jurisdiction   = $jurisdiction
    """, **d)
print("Tạo Document node xong")

Tạo Document node xong


In [7]:
# Cell 4 — Tạo Activity nodes
with driver.session() as session:
    for act in reg['activities']:
        session.run("""
            MERGE (a:Activity {id: $id})
            SET a.name         = $name,
                a.description  = $description,
                a.event_origin = $event_origin,
                a.article_ref  = $article_ref,
                a.required     = $required
            WITH a
            MATCH (doc:Document {id: $doc_id})
            MERGE (a)-[:DEFINED_IN]->(doc)
        """, doc_id=reg['document']['id'], **act)
print(f"Tạo {len(reg['activities'])} Activity node xong")

Tạo 13 Activity node xong


In [8]:
# Cell 5 — Tạo MUST_PRECEDE edges
with driver.session() as session:
    for seq in reg['sequences']:
        session.run("""
            MATCH (a1:Activity {id: $from_id})
            MATCH (a2:Activity {id: $to_id})
            MERGE (a1)-[r:MUST_PRECEDE]->(a2)
            SET r.article_ref  = $article_ref,
                r.description  = $description,
                r.sequence_id  = $id
        """, from_id=seq['from'],
             to_id=seq['to'],
             **{k: v for k, v in seq.items()
                if k not in ['from', 'to']})
print(f"Tạo {len(reg['sequences'])} MUST_PRECEDE edge xong")

Tạo 7 MUST_PRECEDE edge xong


In [10]:
# Cell 6 — Tạo Condition nodes
with driver.session() as session:
    for cond in reg['conditions']:
        session.run("""
            MERGE (c:Condition {id: $id})
            SET c.description    = $description,
                c.expression     = $expression,
                c.article_ref    = $article_ref,
                c.violation_type = $violation_type
            WITH c
            MATCH (doc:Document {id: $doc_id})
            MERGE (c)-[:DEFINED_IN]->(doc)
        """, doc_id=reg['document']['id'],
             **{k: v for k, v in cond.items()
                if k not in ['threshold', 'unit']})
print(f"Tạo {len(reg['conditions'])} Condition node xong")

Tạo 4 Condition node xong


In [12]:
# Cell 7 — Tạo Role nodes và PERFORMED_BY edges
with driver.session() as session:
    for role in reg['roles']:
        # Tạo Role node
        session.run("""
            MERGE (r:Role {id: $id})
            SET r.name = $name,
                r.type = $type
        """, id=role['id'],
             name=role['name'],
             type=role['type'])

        # Tạo PERFORMED_BY từ Activity → Role
        for act_id in role['performs']:
            session.run("""
                MATCH (a:Activity {id: $act_id})
                MATCH (r:Role {id: $role_id})
                MERGE (a)-[:PERFORMED_BY]->(r)
            """, act_id=act_id, role_id=role['id'])

print(f"Tạo {len(reg['roles'])} Role node xong")

Tạo 3 Role node xong


In [13]:
# Cell 8 — Kiểm tra Regulation Graph
with driver.session() as session:
    r1 = session.run(
        "MATCH (n:Activity) RETURN count(n) AS cnt"
    )
    r2 = session.run(
        "MATCH ()-[r:MUST_PRECEDE]->() RETURN count(r) AS cnt"
    )
    r3 = session.run(
        "MATCH (n:Condition) RETURN count(n) AS cnt"
    )
    r4 = session.run(
        "MATCH (n:Role) RETURN count(n) AS cnt"
    )
    print("Regulation Graph trong Neo4j:")
    print(f"  Activity node   : {r1.single()['cnt']}")
    print(f"  MUST_PRECEDE    : {r2.single()['cnt']}")
    print(f"  Condition node  : {r3.single()['cnt']}")
    print(f"  Role node       : {r4.single()['cnt']}")

Regulation Graph trong Neo4j:
  Activity node   : 13
  MUST_PRECEDE    : 7
  Condition node  : 4
  Role node       : 3


In [14]:
# Cell 9 — Xem graph trong Neo4j Browser
print("Chạy query sau trong Neo4j Browser:")
print("""
MATCH (a1:Activity)-[r:MUST_PRECEDE]->(a2:Activity)
RETURN a1, r, a2
""")

Chạy query sau trong Neo4j Browser:

MATCH (a1:Activity)-[r:MUST_PRECEDE]->(a2:Activity)
RETURN a1, r, a2

